# 原模型诊断
先测重复调用噪声，再测无关上下文与大 K；不预设原模型不稳定。

In [1]:
from pathlib import Path
import os, sys, subprocess, zipfile
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE = Path('/content/drive/MyDrive/SA-Jev')
    PROJECT = Path('/content/graphrag-jev')
except ImportError:
    DRIVE = Path(os.environ.get('SAJEV_WORKDIR', '../outputs/colab_local')).resolve()
    PROJECT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DRIVE.mkdir(parents=True, exist_ok=True)
REPO_URL = os.environ.get('SAJEV_REPO_URL', 'https://github.com/BoldHu/SA-Jev.git')  # Override with SAJEV_REPO_URL if needed.
SYNC_GIT = True
if not PROJECT.exists():
    if REPO_URL:
        subprocess.run(['git', 'clone', REPO_URL, str(PROJECT)], check=True)
    elif (DRIVE / 'sa-jev-source.zip').exists():
        PROJECT.mkdir(parents=True)
        with zipfile.ZipFile(DRIVE / 'sa-jev-source.zip') as archive:
            archive.extractall(PROJECT)
    else:
        raise RuntimeError('Set SAJEV_REPO_URL or copy sa-jev-source.zip to MyDrive/SA-Jev/')
if SYNC_GIT and (PROJECT / '.git').exists():
    status = subprocess.check_output(['git', '-C', str(PROJECT), 'status', '--porcelain'], text=True)
    if status.strip():
        print('Local edits present; preserving them. Git synchronization skipped.')
    else:
        subprocess.run(['git', '-C', str(PROJECT), 'pull', '--ff-only'], check=True)
os.chdir(PROJECT)
MODE = 'full'  # Change to 'full' only after the pilot completes and memory is measured.
ROOT = DRIVE / 'public_lightrag_v1' / MODE
ROOT.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(DRIVE / 'huggingface')
PUBLIC_ASSETS = DRIVE / 'public_lightrag_assets'
CONFIG_PATH = PROJECT / 'configs/public_lightrag.yaml'
print('Code:', PROJECT, 'Artifacts:', ROOT)


In [2]:
# Run this in each fresh Colab runtime. No model weights are downloaded here.
subprocess.run([sys.executable, 'scripts/bootstrap_upstream.py'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.[train,retrieval,test]'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-deps', '-e', 'vendor/OpenJev'], check=True)
if 'google.colab' in sys.modules:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'fla-core==0.5.2'], check=True)
# Activate editable-install .pth files in this already-running notebook kernel.
import importlib, site
site_dirs = site.getsitepackages()
if site.ENABLE_USER_SITE:
    site_dirs.append(site.getusersitepackages())
for site_dir in dict.fromkeys(site_dirs):
    site.addsitedir(site_dir)
importlib.invalidate_caches()
import sa_jev
print('SA-Jev source:', sa_jev.__file__)
# If package installation asks for a runtime restart, restart before importing torch.
from sa_jev.experiments import load_config, environment_manifest
config = load_config(CONFIG_PATH)
environment_manifest(ROOT / 'environment.json', config)
def cli(command, *args):
    from collections import deque
    tail = deque(maxlen=40)
    cmd = [sys.executable, '-u', '-m', 'sa_jev.cli', command, '--root', str(ROOT), '--config', str(CONFIG_PATH), *map(str,args)]
    with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, encoding='utf-8', errors='replace', bufsize=1) as process:
        try:
            for line in process.stdout:
                print(line, end='', flush=True)
                tail.append(line)
            returncode = process.wait()
        except KeyboardInterrupt:
            process.terminate()
            process.wait()
            raise
    if returncode:
        raise RuntimeError(f'sajev {command} failed (exit {returncode}). Last output: ' + ''.join(tail))


In [3]:
# Read the same credential file in every notebook. Fill it in once.
from sa_jev.environment import load_environment
ENV_FILE = (DRIVE if 'google.colab' in sys.modules else PROJECT) / '.env'
credential_status = load_environment(ENV_FILE, template=PROJECT / '.env.example')
print('Credential file:', ENV_FILE)
for name in ('OPENAI_API_KEY', 'OPENAI_BASE_URL', 'TYPESAFE_API_KEY', 'HF_TOKEN'):
    print(name + ':', 'loaded' if credential_status[name] else 'empty')
if not credential_status['OPENAI_API_KEY']:
    print('Fill OPENAI_API_KEY in the file above, save it, then rerun this cell before building the index or answering questions.')


先核对 OpenJev 完整产物：固定 revision 的 manifest、各文件大小和 SHA-256。只有缺失/损坏文件才重下，坏缓存入口保留为 .invalid-*；不删除 00 数据或检索缓存。Colab 从 Drive 缓存复制到本机临时目录，以普通文件加载；约需额外 9 GB 本地磁盘，复制有进度，重新连接需要重新复制，通常不需要重新下载。此格不分配 4B GPU 模型。

In [4]:
import importlib
from sa_jev import artifact_cache, model, experiments
for module in (artifact_cache, model, experiments):
    importlib.reload(module)
verified_artifact = artifact_cache.prepare_openjev_artifact(config['model']['artifact'], config['model'].get('revision'))
print('Verified OpenJev artifact:', verified_artifact)

Keep the prepared full data root. Start with 2 questions; set RUN_FULL_DIAGNOSTICS=True only after status=ok and measured latency/memory are available. Run repeat diagnostics and K efficiency in separate cells.

In [ ]:
# Keep MODE/ROOT='full' to reuse the completed data; control run size here.
import copy, importlib, torch
from sa_jev import io, model, evaluation, experiments
for module in (io, model, evaluation, experiments):
    importlib.reload(module)
RUN_FULL_DIAGNOSTICS = False  # Set True after the 2-question check succeeds.
DIAGNOSTIC_LIMIT = (8 if MODE == 'pilot' else 100) if RUN_FULL_DIAGNOSTICS else 2
efficiency_config = copy.deepcopy(config)
if not RUN_FULL_DIAGNOSTICS:
    efficiency_config['evaluation']['k_values'] = [16]
print('ROOT:', ROOT, 'questions:', DIAGNOSTIC_LIMIT,
      'repeats:', config['evaluation']['repeats'],
      'efficiency K:', efficiency_config['evaluation']['k_values'], flush=True)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0),
          'native BF16:', torch.cuda.is_bf16_supported(including_emulation=False), flush=True)
    if not torch.cuda.is_bf16_supported(including_emulation=False):
        print('This BF16 recipe is not validated on T4. Prefer A100 for the 4B experiments.', flush=True)
else:
    print('No CUDA GPU detected. Select a GPU runtime for OpenJev-4B.', flush=True)


In [ ]:
from sa_jev.experiments import diagnostics_experiment, efficiency_experiment
diagnostic_rows = diagnostics_experiment(ROOT, config, limit=DIAGNOSTIC_LIMIT)

In [ ]:
efficiency_rows = efficiency_experiment(ROOT, efficiency_config, limit=DIAGNOSTIC_LIMIT)

In [ ]:
if os.environ.get('TYPESAFE_API_KEY') or os.environ.get('JEV_API_KEY'):
    diagnostics_experiment(ROOT, config, method='dynamic_jev', limit=DIAGNOSTIC_LIMIT)
    efficiency_experiment(ROOT, efficiency_config, method='dynamic_jev', limit=DIAGNOSTIC_LIMIT)
else:
    print('Official Jev API unavailable; no official-Jev result is reported.')

无关状态干扰使用独立且经过核验的测试包，保持不调参。未核验草稿不会被自动当成测试结果。

In [ ]:
from sa_jev.experiments import robustness_experiment
if (ROOT/'data/perturbations_reviewed.jsonl').exists():
    robustness_experiment(ROOT, config, tag='original_perturbations', methods=['openjev_frozen'] + (['dynamic_jev'] if os.environ.get('TYPESAFE_API_KEY') else []))
else:
    print('Awaiting reviewed independent perturbation packets; repeat/K diagnostics above are separate measurements.')